# 05 - KAN-GRU-ATT (HybridTriNet) on our dataset

Re-write of `Train_KAN_GRU_ATT_With_new_dataset_score`, adapted to run on **our** data
(`data/processed/clean_data_exo_ver1.csv`) instead of the Google-Drive Excel + news DB.

What changed:
- Reads the local CSV (no Colab / Drive / openpyxl).
- Reads local scored news from `data/news/archive_2026-06-23/combined_news_scored.csv`.
- Builds daily topic/count/sentiment/intensity features from the scored news archive.
- Derives author-style `news_abnormal`, `impact_score`, and `trend` from daily scored sentiment, then shifts news features by one day.
- GPU is used automatically when available (`cuda`), else CPU.
- The model/training loop remains the original: GRU + Transformer-Attention `HybridTriNet`, delta-target prediction, K=32 window, z-score scaling, EMA, mixed delta+MAPE loss, early stopping on val MAPE.

Run all cells again after this change to refresh the metrics; previous outputs were from the no-news-feature version.


In [1]:
import os, json, time, warnings
warnings.filterwarnings("ignore")
from dataclasses import dataclass
from typing import List, Tuple
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

CSV_PATH         = "../data/processed/clean_data_exo_ver1.csv"
NEWS_SCORED_PATH = "../data/news/archive_2026-06-23/combined_news_scored.csv"
DATE_COL         = "Ng\u00e0y"
TARGET_COLS      = ["MG95", "MG92", "DO 0.001%", "DO 0.05%"]
OUT_DIR          = "../results/kan_gru_att"
os.makedirs(OUT_DIR, exist_ok=True)


Device: cuda


## Config (same as the original notebook)

In [2]:
@dataclass
class CFG:
    K: int = 32
    STEP: int = 1
    val_size: int = 60
    test_size: int = 22
    batch_size: int = 64
    epochs: int = 120
    lr: float = 5e-4
    weight_decay: float = 1e-4
    lr_factor: float = 0.5
    lr_patience: int = 12
    lr_threshold: float = 1e-4
    min_lr: float = 1e-6
    patience: int = 20
    min_delta: float = 1e-4
    use_news_features: bool = False
    shift_news_days: int = 1
    news_shock_abs: float = 0.8
    news_shock_abnormal: float = 0.8
    warmup_epochs: int = 3
    loss_alpha: float = 0.2
    loss_mape_weight: float = 1.0
    mape_eps: float = 1e-3
    clip_grad: float = 5.0

cfg = CFG()


## Utils — features, split, metrics

In [3]:
def set_seed(s=42):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def norm_cols(df):
    df = df.copy()
    df.columns = df.columns.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    return df

def add_calendar_features(df, date_col):
    df = df.copy(); d = pd.to_datetime(df[date_col])
    df["dow"]=d.dt.weekday.astype(float); df["month"]=d.dt.month.astype(float)
    df["year"]=d.dt.year.astype(float);   df["dom"]=d.dt.day.astype(float)
    return df

def _parse_day(s):
    return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()

def _abs_sum(s):
    return float(np.abs(pd.to_numeric(s, errors="coerce").fillna(0.0)).sum())

def _author_style_daily_news(g, shock_abs=0.8, shock_abnormal=0.8):
    # Reference notebook logic adapted from impact_score/news_abnormal to this CSV's sentiment column.
    imp = pd.to_numeric(g["sentiment"], errors="coerce").dropna()
    abnormal_max = float(imp.abs().max()) if len(imp) else 0.0

    if imp.empty:
        impact_base = 0.0
        shock_val = 0.0
    else:
        impact_base = float(imp.quantile([0.25, 0.50, 0.75]).mean())
        shock_val = float(imp.loc[imp.abs().idxmax()])

    use_shock = (abs(shock_val) >= float(shock_abs)) or (abnormal_max >= float(shock_abnormal))
    impact = shock_val if use_shock else impact_base

    if impact > 0:
        trend = "INCREASE"
    elif impact < 0:
        trend = "DECREASE"
    else:
        trend = "NEUTRAL"

    return pd.Series({
        "news_abnormal": abnormal_max,
        "impact_score": float(impact),
        "trend": trend,
    })

def build_daily_news_features(news_path, shock_abs=0.8, shock_abnormal=0.8):
    news = norm_cols(pd.read_csv(news_path))
    date_source = "date" if "date" in news.columns else "datetime"
    required = {date_source, "topic", "sentiment"}
    missing = sorted(required - set(news.columns))
    if missing:
        raise ValueError(f"News file is missing columns: {missing}")

    news["date"] = _parse_day(news[date_source])
    news["topic"] = news["topic"].astype(str).str.strip()
    news["sentiment"] = pd.to_numeric(news["sentiment"], errors="coerce")
    news = news.dropna(subset=["date", "sentiment"]).copy()
    if news.empty:
        return pd.DataFrame(columns=["date", "news_abnormal", "impact_score", "trend"])

    daily_index = pd.DataFrame({"date": pd.date_range(news["date"].min(), news["date"].max(), freq="D")})

    topic_daily = (
        news.groupby(["date", "topic"])["sentiment"]
            .agg(n="count", sent_mean="mean", sent_sum="sum", intensity=_abs_sum)
            .reset_index()
    )
    topic_wide = topic_daily.pivot(index="date", columns="topic", values=["n", "sent_mean", "sent_sum", "intensity"])
    topic_wide.columns = [f"{topic}_{metric}" for metric, topic in topic_wide.columns]
    topic_wide = topic_wide.reset_index()

    all_daily = (
        news.groupby("date")["sentiment"]
            .agg(all_n="count", all_sent_mean="mean", all_sent_sum="sum", all_intensity=_abs_sum)
            .reset_index()
    )
    author_daily = (
        news.groupby("date")
            .apply(lambda g: _author_style_daily_news(g, shock_abs=shock_abs, shock_abnormal=shock_abnormal))
            .reset_index()
    )

    daily = daily_index.merge(topic_wide, on="date", how="left").merge(all_daily, on="date", how="left").merge(author_daily, on="date", how="left")
    numeric_cols = [c for c in daily.columns if c not in ("date", "trend")]
    daily[numeric_cols] = daily[numeric_cols].fillna(0.0)
    daily["trend"] = daily["trend"].fillna("NEUTRAL")
    return daily

def prep_news(df, shift_days=1, news_cols=None):
    df = df.copy()
    if news_cols is None:
        news_cols = ["news_abnormal", "impact_score"]
    for c in news_cols:
        if c not in df.columns:
            df[c] = 0.0
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    if shift_days > 0 and news_cols:
        df[news_cols] = df[news_cols].shift(shift_days).fillna(0.0)
    return df

def merge_news_features(df, date_col, news_path, shift_days=1, shock_abs=0.8, shock_abnormal=0.8):
    daily = build_daily_news_features(news_path, shock_abs=shock_abs, shock_abnormal=shock_abnormal)
    if daily.empty:
        return prep_news(df, shift_days=shift_days)

    out = df.merge(daily, how="left", left_on=date_col, right_on="date").drop(columns=["date"])
    news_cols = [c for c in daily.columns if c not in ("date", "trend")]
    for c in news_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
    out["trend"] = out["trend"].fillna("NEUTRAL")
    out = prep_news(out, shift_days=shift_days, news_cols=news_cols)
    print(
        f"merged {len(news_cols)} numeric news features "
        f"from {daily['date'].min().date()} to {daily['date'].max().date()} "
        f"| shifted by {shift_days} day(s)"
    )
    return out

def fill_missing(df, date_col):
    df = df.copy()
    for c in df.columns:
        if c != date_col and pd.api.types.is_numeric_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], errors="coerce").ffill().bfill()
    return df

def time_split(T, K, val_size, test_size):
    te_e=T; te_s=max(0,T-test_size); va_e=te_s; va_s=max(0,va_e-val_size); tr_e=max(va_s,K+1)
    return (0,tr_e),(va_s,va_e),(te_s,te_e)

def mape(yp, yt, eps=1e-6):
    if yp is None or len(yp)==0: return float("nan")
    yp=np.asarray(yp,np.float32); yt=np.asarray(yt,np.float32)
    return float(np.mean(np.abs(yp-yt)/(np.abs(yt)+eps))*100)

def per_target_metrics(Yp, Yt):
    out={}
    for j,c in enumerate(TARGET_COLS):
        yp,yt=Yp[:,j],Yt[:,j]; ss=float(np.sum((yt-yt.mean())**2))
        out[c]=dict(MAE=round(float(np.mean(np.abs(yp-yt))),4),
                    RMSE=round(float(np.sqrt(np.mean((yp-yt)**2))),4),
                    MAPE=round(float(np.mean(np.abs(yp-yt)/(np.abs(yt)+1e-6))*100),4),
                    SMAPE=round(float(np.mean(2*np.abs(yp-yt)/(np.abs(yp)+np.abs(yt)+1e-6))*100),4),
                    R2=round(float(1-np.sum((yt-yp)**2)/(ss+1e-12)),4))
    return out


## Dataset, model (HybridTriNet = GRU + Transformer-Attention), EMA

In [4]:
class WindowDeltaDataset(Dataset):
    def __init__(self, Xs, Ys, K, start, end, step=1):
        self.Xs, self.Ys, self.K = Xs, Ys, int(K)
        self.t_list = list(range(max(self.K, start), end, max(1, int(step))))
    def __len__(self): return len(self.t_list)
    def __getitem__(self, i):
        t = self.t_list[i]
        return torch.from_numpy(self.Xs[t-self.K:t]).float(), torch.from_numpy(self.Ys[t]).float()

class HybridTriNet(nn.Module):
    def __init__(self, k, D_in, H, D_out, gru_hidden=96, gru_layers=1,
                 attn_dmodel=48, attn_heads=4, attn_layers=1, attn_drop=0.35):
        super().__init__()
        self.in_norm = nn.LayerNorm(D_in)
        self.gru = nn.GRU(D_in, gru_hidden, gru_layers, batch_first=True, bidirectional=False)
        self.proj = nn.Linear(D_in, attn_dmodel)
        enc = nn.TransformerEncoderLayer(d_model=attn_dmodel, nhead=attn_heads,
                                         dim_feedforward=max(128, attn_dmodel*4),
                                         dropout=attn_drop, batch_first=True, activation="gelu")
        self.attn = nn.TransformerEncoder(enc, num_layers=attn_layers)
        self.head = nn.Sequential(nn.Linear(gru_hidden+attn_dmodel, 128), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(128, D_out))
    def forward(self, x):
        x = self.in_norm(x)
        h_gru = self.gru(x)[0][:, -1, :]
        h_attn = self.attn(self.proj(x))[:, -1, :]
        return self.head(torch.cat([h_gru, h_attn], dim=-1))

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay=decay; self.backup=None
        self.shadow={k:v.detach().clone() for k,v in model.state_dict().items() if torch.is_tensor(v)}
    @torch.no_grad()
    def update(self, model):
        for k,v in model.state_dict().items():
            if torch.is_tensor(v):
                if k in self.shadow: self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
                else: self.shadow[k]=v.detach().clone()
    @torch.no_grad()
    def apply(self, model):
        self.backup={k:v.detach().clone() for k,v in model.state_dict().items() if torch.is_tensor(v)}
        for k,v in model.state_dict().items():
            if torch.is_tensor(v): v.copy_(self.shadow[k])
    @torch.no_grad()
    def restore(self, model):
        if self.backup is None: return
        for k,v in model.state_dict().items():
            if torch.is_tensor(v) and k in self.backup: v.copy_(self.backup[k])
        self.backup=None

@torch.no_grad()
def collect_price_preds(model, Xs, df, fc, K, start, end):
    ti=[fc.index(c) for c in TARGET_COLS]
    xmt=torch.tensor(X_MEAN[ti],device=DEVICE); xst=torch.tensor(X_STD[ti],device=DEVICE)
    ymt=torch.tensor(Y_MEAN,device=DEVICE);     yst=torch.tensor(Y_STD,device=DEVICE)
    P,Tr=[],[]
    for t in range(max(K,start), end):
        xb=torch.from_numpy(Xs[t-K:t]).float().unsqueeze(0).to(DEVICE)
        pp=(xb[:,-1,ti]*xst+xmt)+(model(xb)*yst+ymt)
        P.append(pp.squeeze(0).cpu().numpy()); Tr.append(df.loc[t,TARGET_COLS].values.astype(np.float32))
    if not P: return np.zeros((0,4),np.float32), np.zeros((0,4),np.float32)
    return np.stack(P).astype(np.float32), np.stack(Tr).astype(np.float32)

## Load data, build features, split, scale

In [5]:
set_seed(42)
df = norm_cols(pd.read_csv(CSV_PATH))
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
df = add_calendar_features(df, DATE_COL)
if cfg.use_news_features:
    df = merge_news_features(
        df,
        DATE_COL,
        NEWS_SCORED_PATH,
        shift_days=cfg.shift_news_days,
        shock_abs=cfg.news_shock_abs,
        shock_abnormal=cfg.news_shock_abnormal,
    )
else:
    df = prep_news(df, cfg.shift_news_days)
df = fill_missing(df, DATE_COL)
for c in TARGET_COLS: df[c] = df[c].ffill().bfill()

# delta targets
Yd = np.stack([df[c].diff().fillna(0.0).values.astype(np.float32) for c in TARGET_COLS], axis=1)

feature_cols = [c for c in df.columns if c not in (DATE_COL, "trend") and pd.api.types.is_numeric_dtype(df[c])]
for c in TARGET_COLS:
    if c not in feature_cols: feature_cols.append(c)
news_feature_cols = [c for c in feature_cols if c in ("news_abnormal", "impact_score") or c.endswith(("_n", "_sent_mean", "_sent_sum", "_intensity"))]

X = df[feature_cols].values.astype(np.float32); T, D_in = X.shape
(tr0,tr1),(va0,va1),(te0,te1) = time_split(T, cfg.K, cfg.val_size, cfg.test_size)
print(
    f"T={T} D_in={D_in} K={cfg.K} | train={tr1-tr0} val={va1-va0} test={te1-te0} "
    f"| features={len(feature_cols)} | news_features={len(news_feature_cols)}"
)
print("news feature columns:", news_feature_cols)

# z-score (fit on train only); delta z-score too
X_MEAN = X[:tr1].mean(0).astype(np.float32); X_STD = (X[:tr1].std(0)+1e-8).astype(np.float32)
Xs = (X - X_MEAN) / X_STD
Y_MEAN = Yd[:tr1].mean(0).astype(np.float32); Y_STD = (Yd[:tr1].std(0)+1e-8).astype(np.float32)
Ys = (Yd - Y_MEAN) / Y_STD

dl_tr = DataLoader(WindowDeltaDataset(Xs, Ys, cfg.K, tr0, tr1, cfg.STEP), batch_size=cfg.batch_size, shuffle=True)


T=4649 D_in=23 K=32 | train=4567 val=60 test=22 | features=23 | news_features=2
news feature columns: ['news_abnormal', 'impact_score']


## Train (early stop on val MAPE) + evaluate on test

In [6]:
set_seed(42)
ti = [feature_cols.index(c) for c in TARGET_COLS]
xmt=torch.tensor(X_MEAN[ti],device=DEVICE); xst=torch.tensor(X_STD[ti],device=DEVICE)
ymt=torch.tensor(Y_MEAN,device=DEVICE);     yst=torch.tensor(Y_STD,device=DEVICE)

model = HybridTriNet(cfg.K, D_in, 1, 4).to(DEVICE)
ema = EMA(model, 0.999)
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, "min", factor=cfg.lr_factor,
        patience=cfg.lr_patience, threshold=cfg.lr_threshold, min_lr=cfg.min_lr)

best=float("inf"); best_ep=0; bad=0; va_ema=None
ck = os.path.join(OUT_DIR, "best.pth")
for ep in range(cfg.epochs):
    model.train()
    if ep < cfg.warmup_epochs:
        for pg in opt.param_groups: pg["lr"] = cfg.lr*(ep+1)/max(1,cfg.warmup_epochs)
    for xb, yb in dl_tr:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        ps = model(xb)
        delta_loss = torch.mean(torch.abs(ps - yb))
        lp = xb[:,-1,ti]*xst+xmt
        pp = lp + ps*yst+ymt; tp = lp + yb*yst+ymt
        mape_loss = torch.mean(torch.abs(pp-tp)/(torch.abs(tp)+cfg.mape_eps))
        loss = cfg.loss_alpha*delta_loss + cfg.loss_mape_weight*mape_loss
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad); opt.step(); ema.update(model)
    ema.apply(model); model.eval()
    Yp, Yt = collect_price_preds(model, Xs, df, feature_cols, cfg.K, va0, va1)
    vm = mape(Yp, Yt)
    va_ema = vm if va_ema is None else 0.9*va_ema + 0.1*vm
    if vm < best - cfg.min_delta: best=vm; best_ep=ep+1; bad=0; torch.save(model.state_dict(), ck)
    else: bad += 1
    ema.restore(model)
    if ep >= cfg.warmup_epochs and np.isfinite(va_ema): sch.step(va_ema)
    if (ep+1) % 5 == 0:
        print(f"ep {ep+1:3d} | valMAPE={vm:.3f} best={best:.3f}@{best_ep} lr={opt.param_groups[0]['lr']:.1e}")
    if bad >= cfg.patience:
        print(f"early stop @ {ep+1}, best valMAPE={best:.3f}% @ {best_ep}"); break

# evaluate best checkpoint on val + test
model.load_state_dict(torch.load(ck, map_location=DEVICE))
Yp_va, Yt_va = collect_price_preds(model, Xs, df, feature_cols, cfg.K, va0, va1)
Yp_te, Yt_te = collect_price_preds(model, Xs, df, feature_cols, cfg.K, te0, te1)

print(f"\n=== RESULTS (test set, {te1-te0} days) | best epoch {best_ep} ===")
print("overall test MAPE: %.4f%%" % mape(Yp_te, Yt_te))
import pandas as _pd
res_te = per_target_metrics(Yp_te, Yt_te)
display(_pd.DataFrame(res_te).T)
json.dump({"best_epoch":best_ep, "test_overall_mape":mape(Yp_te,Yt_te),
           "test_per_target":res_te, "val_per_target":per_target_metrics(Yp_va,Yt_va)},
          open(os.path.join(OUT_DIR,"results.json"),"w"), indent=2)
print("saved ->", os.path.join(OUT_DIR,"results.json"))

ep   5 | valMAPE=4.000 best=4.000@5 lr=5.0e-04
ep  10 | valMAPE=3.997 best=3.996@8 lr=5.0e-04
ep  15 | valMAPE=4.010 best=3.996@8 lr=5.0e-04
ep  20 | valMAPE=4.023 best=3.996@8 lr=5.0e-04
ep  25 | valMAPE=4.036 best=3.996@8 lr=2.5e-04
early stop @ 28, best valMAPE=3.996% @ 8

=== RESULTS (test set, 22 days) | best epoch 8 ===
overall test MAPE: 3.1711%


,MAE,RMSE,MAPE,SMAPE,R2
MG95,3.3276,4.0502,2.5420,2.5497,0.2265
MG92,3.3471,4.2697,2.6227,2.6445,0.3125
DO 0.001%,6.5133,8.7326,3.8036,3.7509,0.7402
DO 0.05%,5.6893,7.4846,3.7160,3.6702,0.6213


saved -> ../results/kan_gru_att\results.json


## Notes

- This model predicts the **daily change (delta)** and reconstructs the price; metrics are on the
  reconstructed price level over the last 22 trading days (the original notebook's `test_size`).
- It converges very early on this dataset (best val MAPE at epoch 5; early-stopped at 25).
- News features are 0 here because `clean_data_exo_ver1.csv` has no `impact_score` / `news_abnormal`.
  To add the news signal, merge the Gemini-scored daily features (see `.agent/feature_engineering.md`)
  before training.
- On your GPU this runs in well under a minute (`Run All`).